# Lesson 0.3: Population vs Sample

See that a sample statistic wanders around the fixed population parameter, that bigger samples wander less, and that a biased sample does not.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. The lesson's ten shops

Here the population is small enough to see: monthly sales of 10 shops. The population mean (a parameter) is fixed. Each sample of 3 shops gives a different sample mean (a statistic). The population SD divides by N (ddof=0), the sample SD by n - 1 (ddof=1).

In [1]:
import numpy as np

shops = np.array([12, 15, 9, 22, 18, 7, 25, 14, 20, 11])
mu = shops.mean()
print(f'population mean (mu) = {mu:.2f}, sigma = {shops.std(ddof=0):.3f}')

samples = {'A': shops[[0, 3, 5]], 'B': shops[[6, 4, 1]], 'C': shops[[2, 9, 7]]}
for name, s in samples.items():
    print(f'sample {name}: {s}  x-bar = {s.mean():.2f}  gap = {s.mean() - mu:+.2f}')
print(f'sample A sd = {samples["A"].std(ddof=1):.3f}')

population mean (mu) = 15.30, sigma = 5.551
sample A: [12 22  7]  x-bar = 13.67  gap = -1.63
sample B: [25 18 15]  x-bar = 19.33  gap = +4.03
sample C: [ 9 11 14]  x-bar = 11.33  gap = -3.97
sample A sd = 7.638


## 2. A bigger, seeded population

Now a population of 10,000 weekly study times in hours, made with a fixed seed so everyone gets the same numbers. We pretend not to know its mean and compute it once: this is the parameter.

In [2]:
rng = np.random.default_rng(42)
population = rng.normal(loc=14, scale=4, size=10_000).round(1)

mu = population.mean()
print('population size N =', len(population))
print(f'population mean mu = {mu:.3f}')

population size N = 10000
population mean mu = 13.959


## 3. Sampling error shrinks as n grows

For each sample size we draw one random sample and show its gap from mu. Then we repeat the draw 300 times to see the typical gap. Bigger samples miss by less. The gap never vanishes completely: that is sampling error, and it is normal.

In [3]:
for n in [10, 50, 250, 1000]:
    one = rng.choice(population, size=n, replace=False).mean()
    many = [rng.choice(population, size=n, replace=False).mean() for _ in range(300)]
    typical = np.mean(np.abs(np.array(many) - mu))
    print(f'n = {n:4d}: one sample mean = {one:6.2f}, typical gap over 300 samples = {typical:.3f}')

n =   10: one sample mean =  12.40, typical gap over 300 samples = 1.066
n =   50: one sample mean =  14.58, typical gap over 300 samples = 0.473
n =  250: one sample mean =  14.04, typical gap over 300 samples = 0.203
n = 1000: one sample mean =  13.80, typical gap over 300 samples = 0.098


## 4. A biased sample does not get better

Suppose we only ask the 1,000 heaviest studiers (say, people we meet in the library). Even with a large sample the mean stays far from mu, because the method is tilted. A bigger sample cannot fix bias.

In [4]:
top = np.sort(population)[-1000:]

for n in [10, 100, 1000]:
    m = rng.choice(top, size=n, replace=False).mean()
    print(f'biased sample n = {n:4d}: mean = {m:.2f}, gap from mu = {m - mu:+.2f}')

biased sample n =   10: mean = 21.05, gap from mu = +7.09
biased sample n =  100: mean = 20.79, gap from mu = +6.83
biased sample n = 1000: mean = 21.05, gap from mu = +7.09


## Try this

- Change the seed from 42 to 7 and rerun. Which numbers change, and does the pattern (gap shrinking with n) stay the same?
- In the biased block, take the top 3,000 values instead of the top 1,000. Predict whether the gap gets smaller or bigger.

## The same idea in R, SPSS and Stata

**R**

```r
shops <- c(12, 15, 9, 22, 18, 7, 25, 14, 20, 11)
mean(shops)
sqrt(mean((shops - mean(shops))^2))
A <- shops[c(1, 4, 6)]
mean(A)
sd(A)
mean(shops[c(7, 5, 2)])
mean(shops[c(3, 10, 8)])
```

**SPSS** (Analyze > Descriptive Statistics > Descriptives)

```
DATA LIST FREE / sales inA.
BEGIN DATA
12 1 15 0 9 0 22 1 18 0
7 1 25 0 14 0 20 0 11 0
END DATA.
DESCRIPTIVES VARIABLES=sales.
TEMPORARY.
SELECT IF (inA = 1).
DESCRIPTIVES VARIABLES=sales.
```

**Stata**

```
clear
set obs 10
generate sales = real(word("12 15 9 22 18 7 25 14 20 11", _n))
generate inA = inlist(_n, 1, 4, 6)
summarize sales
summarize sales if inA == 1
```